In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality, data_reader
from food_security.fao_api import FAOClient

from pathlib import Path

In [2]:
config = dotenv.dotenv_values(".env")
username = config["FAOSTAT_USERNAME"]
password = config["FAOSTAT_PASSWORD"]

fao_client = FAOClient(username=username, password=password)

In [3]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [4]:
wq_his_file = src_dir / 'JCARWQV7.Rbd/10/advirrig.his'

wq_his_file = data_reader.HisFile(wq_his_file, crop=None)
wq_his_file.read(hia=True)
wq_ds = wq_his_file.ds.copy(deep=True)

In [5]:
toml_file = src_dir.parent / "salinity_correction_egypt.toml"
water_prod_df, water_use_df = water_quality.generate_water_csv(
    config_path=toml_file,
    # corrected_df=corrected_df,
    save=True,
    fao_client=fao_client,
)

excel_water_prod_df = pd.DataFrame(
    {
        "area": water_prod_df["area_map_name"],
        "year": water_prod_df["year"],
        "water_productivity": water_prod_df["water_productivity"],
        "hectares": water_prod_df["hectares"],
    }
)

excel_water_use_df = pd.DataFrame(
    {
        "area": water_use_df["area_map_name"],
        "year": water_use_df["year"],
        "timestep": water_use_df["timestep"],
        "water_supply": water_use_df["water_supply"],
        "water_demand": water_use_df["water_demand"],
        "water_use": water_use_df["water_use"],
        "water_exploitation_index": water_use_df["water_exploitation_index"],
        "hectares": water_use_df["hectares"],
    }
)

config input salinity_correction.crop_production.path contains a non-existing path
config input salinity_correction.mapping.path contains a non-existing path
2026-08-05 11:04:00,588 | WARNING  | root | config input salinity_correction.crop_production.path contains a non-existing path
2026-08-05 11:04:00,588 | WARNING  | root | config input salinity_correction.mapping.path contains a non-existing path
2026-08-05 11:04:01,908 | INFO     | food_security.salinity_correction | Starting crop yield correction for Egypt
2026-08-05 11:04:01,910 | INFO     | food_security.salinity_correction | Loaded input data. Areas=68, Crops=31, Years=2


Areas:   0%|          | 0/68 [00:00<?, ?it/s]

2026-08-05 11:04:26,442 | INFO     | food_security.salinity_correction | Created dataframe with 3226 rows
2026-08-05 11:04:26,443 | INFO     | food_security.salinity_correction | Crop yield correction finished successfully
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/food_security/water_quality.py:109: RuntimeWarning: invalid value encountered in scalar divide
  water_productivity = producer_price / water_supply_annual
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/food_security/water_quality.py:149: RuntimeWarning: invalid value encountered in divide
  water_expoitation_index = water_supply / water_demand
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/food_security/water_quality.py:109: RuntimeWarning: invalid value encountered in scalar divide
  water_productivity = producer_price / water_supply_annual
/Users/hemert/projects/egypt-survey-ml/.venv/lib/python3.12/site-packages/food_security/water_quality

In [6]:
excel_water_use_yearly_df = pd.DataFrame(
    {
        "area": excel_water_use_df['area'].unique()
    }
)

water_supply_list = []
water_demand_list = []

for area_name in excel_water_use_df['area'].unique():
    area_df = water_use_df[(water_use_df['area_map_name'] == area_name) & (water_use_df['year'] == 2021)]
    water_supply = area_df['water_supply'].mean() * 365.25 * 24 * 3600
    water_demand = area_df['water_demand'].mean() * 365.25 * 24 * 3600

    water_supply_list.append(water_supply)
    water_demand_list.append(water_demand)

excel_water_use_yearly_df['year'] = 2021
excel_water_use_yearly_df['water_supply'] = water_supply_list
excel_water_use_yearly_df['water_demand'] = water_demand_list

In [7]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)


In [8]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

In [9]:
mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

In [10]:
excel_water_use_df['area'] = excel_water_use_df['area'].map(mapping)
excel_water_use_yearly_df['area'] = excel_water_use_yearly_df['area'].map(mapping)
excel_water_prod_df['area'] = excel_water_prod_df['area'].map(mapping)

In [11]:
for i, row in excel_water_prod_df.iterrows():
    year = row['year']
    area = row['area']
    start_ts = f"{year}-10-01"
    end_ts = f"{year + 1}-10-1"
    timeframe = slice(start_ts, end_ts)

    surface_water = (
        wq_ds.sel({"station": area, "time": timeframe})
        .mean(dim="time")["+ Allocated SW (m3/s)"]
        .values
    )

    ground_water = (
        wq_ds.sel({"station": area, "time": timeframe})
        .mean(dim="time")["+ Allocated GW (m3/s)"]
        .values
    )

    water_ratio = (
        wq_ds.sel({"station": area, "time": timeframe})
        .mean(dim="time")["Demand from network (m3/s)"]
        .values
    )

    excel_water_prod_df.loc[i, 'surface_water_mean'] = surface_water
    excel_water_prod_df.loc[i, 'ground_water_mean'] = ground_water
    excel_water_prod_df.loc[i, 'supply_demand_ratio'] = water_ratio

In [12]:
write_excel_file(excel_water_prod_df, excel_path, sheet_name="water_productivity")
write_excel_file(excel_water_use_yearly_df, excel_path, sheet_name="water_use")

In [13]:
excel_command_df = pd.read_excel(excel_path, sheet_name='command_unit')

excel_command_df = (
    excel_command_df
    .drop(columns=excel_water_prod_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_water_prod_df[excel_water_prod_df['year'] == 2021].drop(columns=['year', 'hectares']), on="area", how="left")
)

excel_command_df = (
    excel_command_df
    .drop(columns=excel_water_use_yearly_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_water_use_yearly_df[excel_water_use_yearly_df['year'] == 2021].drop(columns='year'), on="area", how="left")
)

for column in excel_water_prod_df.columns:
    if column != "area" and column in excel_command_df.columns and column != 'supply_demand_ratio':
        excel_command_df[column] = excel_command_df[column] / excel_command_df['rural_population']

for column in excel_water_use_yearly_df.columns:
    if column != "area" and column in excel_command_df.columns:
        excel_command_df[column] = excel_command_df[column] / excel_command_df['rural_population']

write_excel_file(excel_command_df, excel_path, sheet_name='command_unit')